In [ ]:
rm(list=ls())
library(Seurat)
library(dplyr)
library(ggplot2)
library(cowplot)
library(reshape2)
library(labdsv)
library(lmtest)

In [ ]:
base_path = '/home/EOCRC_atlas/'
date = "DATE_PCoA"

In [ ]:
# load the data 
crc <- readRDS(paste0(base_path, 'data/all_samples_raw_withTier2Annotation_09-19-25.rds'))

# Don't run the mixed marker subsets 
run_celltypes = c('Adipocytes', 'B cell', 'CD4 T cells', 'CD8 T cells',
       'CEACAM1 colonocyte-like', 'Cycing endothelium', 'Cycling Myeloid',
       'Cycling Stromal', 'Cycling T cells', 'Cycling plasma cell', 'DC',
       'Enteroendocrine-like', 'Fibroblast', 'Fibroblast-BMP5-SOX6',
       'Fibroblast-C3', 'Fibroblast-Infl', 'Fibroblast-KCNN3',
       'Fibroblast-MMP2-THY1', 'Germinal center / Cycling B cell',
       'Glial cells', 'HSP-hi Myeloid',
       'HSP-hi Stromal', 'HSP-hi glial', 'ILCs',
       'LGR5 stem cell-like', 'Lymphatic endothelium',
       'MT-Ribo-hi Myeloid', 'MT-Ribo-hi Stromal', 'MT-Ribo-hi T cells',
       'MT-Ribo-hi endothelium', 'MT-Ribo-hi epithelial',
       'MUC2 goblet-like', 'Macrophage-Monocyte', 'Mast',
       'Myofibroblast-SMC', 'NK-Cytotoxic T cells', 'Neuronal cells',
       'Neutrophil', 'Patient-specific', 'Pericytes', 'Plasma cell',
       'Regulatory T cells', 'T helper cells', 'Vascular endothelium')

crc = subset(crc, subset=Annotation_Tier2%in%run_celltypes)

# MSS Tier 1

In [ ]:
# subset crc, calculate dissimilarities, and run pcoa 
df <- subset(crc@meta.data, MSI_v2 == "MSS: STABLE")
print(dim(df))

print(length(unique(df$Annotation_Tier1)))

propmat1 <- dcast(df, Annotation_Tier1 ~ FRID)

propmat <- matrix(as.numeric(unlist(propmat1[,-1])), nrow=nrow(propmat1)) # convert to numeric
rownames(propmat) <- propmat1$Annotation_Tier1
colnames(propmat) <- colnames(propmat1)[-1]

propmat <- sweep(propmat, 2, colSums(propmat), FUN="/")
D <- dsvdis(t(propmat), index="bray/curtis")
dim(as.matrix(D))

pc <- pco(D, 6)

dfc <- df[!duplicated(df$FRID),]
dfc <- dfc[match(colnames(propmat),dfc$FRID),]
dfc$pc1 <- pc$points[,1]
dfc$pc2 <- pc$points[,2]
dfc$pc3 <- pc$points[,3]
dfc$pc4 <- pc$points[,4]
dfc$pc5 <- pc$points[,5]
dfc$pc6 <- pc$points[,6]

varexp <- pc$eig / sum(pmax(0, pc$eig))

In [ ]:
# PCoA plots 
if (!dir.exists(paste0(base_path, "results/", date, "/"))) {dir.create(paste0(base_path, "results/", date, "/"), recursive = TRUE)}

p1 <- ggplot(data=dfc) +
	geom_point(aes(x=pc1, y=pc2, fill=Cohort), shape=21, size=3, color="black", stroke=0.2) + theme_bw() +
	scale_fill_manual(values=c(UnderFifty="#95d962", FiftyPlus="#628cd9")) +
	xlab(sprintf("PCo 1 (%.1f%%)", 100*varexp[1])) +
	ylab(sprintf("PCo 2 (%.1f%%)", 100*varexp[2])) + ggtitle("Cohort") +
	theme(panel.grid.major = element_blank(), panel.grid.minor = element_blank()) + 
    theme(legend.position = "right")


p2 <- ggplot(data=dfc) +
	geom_point(aes(x=pc2, y=pc3, fill=Cohort), shape=21, size=3, color="black", stroke=0.2) + theme_bw() +
	scale_fill_manual(values=c(UnderFifty="#95d962", FiftyPlus="#628cd9")) +
	xlab(sprintf("PCo 2 (%.1f%%)", 100*varexp[2])) +
	ylab(sprintf("PCo 3 (%.1f%%)", 100*varexp[3])) + ggtitle("Cohort") +
	theme(panel.grid.major = element_blank(), panel.grid.minor = element_blank()) + 
    theme(legend.position = "right")


p3 <- ggplot(data=dfc) +
	geom_point(aes(x=pc3, y=pc4, fill=Cohort), shape=21, size=3, color="black", stroke=0.2) + theme_bw() +
	scale_fill_manual(values=c(UnderFifty="#95d962", FiftyPlus="#628cd9")) +
	xlab(sprintf("PCo 3 (%.1f%%)", 100*varexp[3])) +
	ylab(sprintf("PCo 4 (%.1f%%)", 100*varexp[4])) + ggtitle("Cohort") +
	theme(panel.grid.major = element_blank(), panel.grid.minor = element_blank()) + 
    theme(legend.position = "right")


p4 <- ggplot(data=dfc) +
	geom_point(aes(x=pc4, y=pc5, fill=Cohort), shape=21, size=3, color="black", stroke=0.2) + theme_bw() +
	scale_fill_manual(values=c(UnderFifty="#95d962", FiftyPlus="#628cd9")) +
	xlab(sprintf("PCo 4 (%.1f%%)", 100*varexp[4])) +
	ylab(sprintf("PCo 5 (%.1f%%)", 100*varexp[5])) + ggtitle("Cohort") +
	theme(panel.grid.major = element_blank(), panel.grid.minor = element_blank()) + 
    theme(legend.position = "right")

p <- plot_grid(p1, p2, p3, p4, ncol=4)
pdf(paste0(base_path, "results/", date, "/PCoA_Cohort_allSubsets_MSS_Tier1.pdf"), 16, 3)
p
dev.off()

options(repr.plot.width = 16, repr.plot.height = 3)
p

In [ ]:
# look at cell types driving PCo1 
t_propmat = t(propmat)
t_propmat = t_propmat[match(dfc$FRID, rownames(t_propmat)), , drop=FALSE]

pco1_correlations = apply(t_propmat, 2, function(cell_type_abundance) {
  cor(cell_type_abundance, dfc$pc1, method = "spearman")})

pco1_drivers = data.frame(Cell_Type = names(pco1_correlations), Spearman_rho = pco1_correlations)
pco1_drivers = pco1_drivers[order(-pco1_drivers$Spearman_rho), ]
print(pco1_drivers)

In [ ]:
# Look for association wtih age and pco1  
dfc$Age_numeric = as.numeric(as.character((dfc$Age)))
dfc$Age_scaled = as.numeric(scale(dfc$Age_numeric))

lm_model <- lm(pc1 ~ Age_scaled + Sidedness + Sex + Therapy_v2 + Overall_Stage, data = dfc)
summary_stats <- summary(lm_model)
print(summary_stats)

# MSI Tier 1

In [ ]:
# subset crc, calculate dissimilarities, and run pcoa 
df <- subset(crc@meta.data, MSI_v2 == "MSI-H: HIGH")
print(dim(df))

print(length(unique(df$Annotation_Tier1)))

propmat1 <- dcast(df, Annotation_Tier1 ~ FRID)

propmat <- matrix(as.numeric(unlist(propmat1[,-1])), nrow=nrow(propmat1)) # convert to numeric
rownames(propmat) <- propmat1$Annotation_Tier1
colnames(propmat) <- colnames(propmat1)[-1]

propmat <- sweep(propmat, 2, colSums(propmat), FUN="/")
D <- dsvdis(t(propmat), index="bray/curtis")
dim(as.matrix(D))

pc <- pco(D, 6)

dfc <- df[!duplicated(df$FRID),]
dfc <- dfc[match(colnames(propmat),dfc$FRID),]
dfc$pc1 <- pc$points[,1]
dfc$pc2 <- pc$points[,2]
dfc$pc3 <- pc$points[,3]
dfc$pc4 <- pc$points[,4]
dfc$pc5 <- pc$points[,5]
dfc$pc6 <- pc$points[,6]

varexp <- pc$eig / sum(pmax(0, pc$eig))

In [ ]:
# PCoA plot 
if (!dir.exists(paste0(base_path, "results/", date, "/"))) {dir.create(paste0(base_path, "results/", date, "/"), recursive = TRUE)}

p1 <- ggplot(data=dfc) +
	geom_point(aes(x=pc1, y=pc2, fill=Cohort), shape=21, size=3, color="black", stroke=0.2) + theme_bw() +
	scale_fill_manual(values=c(UnderFifty="#95d962", FiftyPlus="#628cd9")) +
	xlab(sprintf("PCo 1 (%.1f%%)", 100*varexp[1])) +
	ylab(sprintf("PCo 2 (%.1f%%)", 100*varexp[2])) + ggtitle("Cohort") +
	theme(panel.grid.major = element_blank(), panel.grid.minor = element_blank()) + 
    theme(legend.position = "right")


p2 <- ggplot(data=dfc) +
	geom_point(aes(x=pc2, y=pc3, fill=Cohort), shape=21, size=3, color="black", stroke=0.2) + theme_bw() +
	scale_fill_manual(values=c(UnderFifty="#95d962", FiftyPlus="#628cd9")) +
	xlab(sprintf("PCo 2 (%.1f%%)", 100*varexp[2])) +
	ylab(sprintf("PCo 3 (%.1f%%)", 100*varexp[3])) + ggtitle("Cohort") +
	theme(panel.grid.major = element_blank(), panel.grid.minor = element_blank()) + 
    theme(legend.position = "right")


p3 <- ggplot(data=dfc) +
	geom_point(aes(x=pc3, y=pc4, fill=Cohort), shape=21, size=3, color="black", stroke=0.2) + theme_bw() +
	scale_fill_manual(values=c(UnderFifty="#95d962", FiftyPlus="#628cd9")) +
	xlab(sprintf("PCo 3 (%.1f%%)", 100*varexp[3])) +
	ylab(sprintf("PCo 4 (%.1f%%)", 100*varexp[4])) + ggtitle("Cohort") +
	theme(panel.grid.major = element_blank(), panel.grid.minor = element_blank()) + 
    theme(legend.position = "right")


p4 <- ggplot(data=dfc) +
	geom_point(aes(x=pc4, y=pc5, fill=Cohort), shape=21, size=3, color="black", stroke=0.2) + theme_bw() +
	scale_fill_manual(values=c(UnderFifty="#95d962", FiftyPlus="#628cd9")) +
	xlab(sprintf("PCo 4 (%.1f%%)", 100*varexp[4])) +
	ylab(sprintf("PCo 5 (%.1f%%)", 100*varexp[5])) + ggtitle("Cohort") +
	theme(panel.grid.major = element_blank(), panel.grid.minor = element_blank()) + 
    theme(legend.position = "right")

p <- plot_grid(p1, p2, p3, p4, ncol=4)
pdf(paste0(base_path, "results/", date, "/PCoA_Cohort_allSubsets_MSI_Tier1.pdf"), 16, 3)
p
dev.off()

options(repr.plot.width = 16, repr.plot.height = 3)
p

In [ ]:
# Find cell types driving PCo1 
t_propmat = t(propmat)
t_propmat = t_propmat[match(dfc$FRID, rownames(t_propmat)), , drop=FALSE]

pco1_correlations = apply(t_propmat, 2, function(cell_type_abundance) {
  cor(cell_type_abundance, dfc$pc1, method = "spearman")})

pco1_drivers = data.frame(Cell_Type = names(pco1_correlations), Spearman_rho = pco1_correlations)
pco1_drivers = pco1_drivers[order(-pco1_drivers$Spearman_rho), ]
print(pco1_drivers)